In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!cp "/content/drive/MyDrive/Forecasting/master_wide_hourly_train_2012_2013.csv" "/content/master_wide_hourly_train_2012_2013.csv"
!cp "/content/drive/MyDrive/Forecasting/master_wide_hourly_validation_2014_01_04.csv" "/content/master_wide_hourly_validation_2014_01_04.csv"
!cp "/content/drive/MyDrive/Forecasting/master_wide_hourly_test_2014_05_12.csv" "/content/master_wide_hourly_test_2014_05_12.csv"
!cp "/content/drive/MyDrive/Forecasting/calendar_features_hourly.csv" "/content/calendar_features_hourly.csv"

In [3]:
!pip install neuralforecast

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.9/269.9 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.2/348.2 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.0/73.0 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 39.1 MB/s eta 0:00:00


In [4]:
from pathlib import Path

# Input files -- wide-format CSVs (timestamp as index, one column per client)
TRAIN_PATH    = "/content/master_wide_hourly_train_2012_2013.csv"
VAL_PATH      = "/content/master_wide_hourly_validation_2014_01_04.csv"
TEST_PATH     = "/content/master_wide_hourly_test_2014_05_12.csv"
CALENDAR_PATH = "/content/calendar_features_hourly.csv"

# Output
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

# Model hyper-parameters
HORIZON       = 24          # predict 24 hours ahead (1 day)
INPUT_SIZE    = 672         # look back 672 hours (4 weeks), same as SARIMAX
MAX_STEPS     = 5000
LEARNING_RATE = 1e-3
BATCH_SIZE    = 32
EARLY_STOP    = 10           # patience in validation checks
VAL_CHECK     = 100         # validate every N training steps
RANDOM_SEED   = 42
SCALER_TYPE   = "standard"  # normalise each series independently

In [5]:
# IMPORTS
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from neuralforecast import NeuralForecast
from neuralforecast.models import iTransformer
from neuralforecast.losses.pytorch import MSE, MAE

In [6]:
# HELPERS

# Calendar exogenous features -- cyclic encodings + is_weekend
EXOG_COLS = ["hour_sin", "hour_cos", "dow_sin", "dow_cos",
             "is_weekend", "month_sin", "month_cos"]

calendar = pd.read_csv(CALENDAR_PATH, parse_dates=["timestamp"])


def wide_to_long(path):
    """Read a wide-format CSV and melt into NeuralForecast long format.

    Columns in wide CSV: timestamp, MT_124, MT_132, ...
    Returns DataFrame with: unique_id, ds, y
    """
    df = pd.read_csv(path, parse_dates=["timestamp"])
    client_cols = [c for c in df.columns if c != "timestamp"]
    long = df.melt(id_vars="timestamp", value_vars=client_cols,
                   var_name="unique_id", value_name="y")
    long = long.rename(columns={"timestamp": "ds"})
    return long.sort_values(["unique_id", "ds"]).reset_index(drop=True)


def merge_calendar(df, calendar, exog_cols):
    """Left-join calendar features onto the long-format DataFrame."""
    return df.merge(calendar[["timestamp"] + exog_cols].rename(
        columns={"timestamp": "ds"}), on="ds", how="left")


print("Helpers ready.")

Helpers ready.


In [7]:
# LOAD & PREPARE DATA
train = merge_calendar(wide_to_long(TRAIN_PATH), calendar, EXOG_COLS)
val   = merge_calendar(wide_to_long(VAL_PATH),   calendar, EXOG_COLS)
test  = merge_calendar(wide_to_long(TEST_PATH),  calendar, EXOG_COLS)

N_SERIES = train["unique_id"].nunique()

print(f"Number of clients (n_series): {N_SERIES}")
print(f"train : {train.shape}  {train['ds'].min()} -> {train['ds'].max()}")
print(f"val   : {val.shape}    {val['ds'].min()} -> {val['ds'].max()}")
print(f"test  : {test.shape}   {test['ds'].min()} -> {test['ds'].max()}")
print()
print(train.head())

Number of clients (n_series): 156
train : (2721888, 10)  2012-01-01 00:00:00 -> 2013-12-31 23:00:00
val   : (445536, 10)    2014-01-01 00:00:00 -> 2014-04-30 23:00:00
test  : (913536, 10)   2014-05-01 00:00:00 -> 2014-12-31 23:00:00

                   ds unique_id           y  hour_sin  hour_cos   dow_sin  \
0 2012-01-01 00:00:00    MT_124  108.851670  0.000000  1.000000 -0.781831   
1 2012-01-01 01:00:00    MT_124  123.205750  0.258819  0.965926 -0.781831   
2 2012-01-01 02:00:00    MT_124  114.832535  0.500000  0.866025 -0.781831   
3 2012-01-01 03:00:00    MT_124  101.674644  0.707107  0.707107 -0.781831   
4 2012-01-01 04:00:00    MT_124  105.263150  0.866025  0.500000 -0.781831   

   dow_cos  is_weekend  month_sin  month_cos  
0  0.62349           1        0.5   0.866025  
1  0.62349           1        0.5   0.866025  
2  0.62349           1        0.5   0.866025  
3  0.62349           1        0.5   0.866025  
4  0.62349           1        0.5   0.866025  


In [8]:
# EVALUATION METRICS
def compute_metrics(df, target_col="y", pred_col="iTransformer"):
    y    = df[target_col].values
    yhat = df[pred_col].values
    mse  = np.mean((y - yhat) ** 2)
    mae  = np.mean(np.abs(y - yhat))
    wape = np.sum(np.abs(y - yhat)) / np.sum(np.abs(y))
    return {"MSE": round(mse, 4), "MAE": round(mae, 4), "WAPE": round(wape, 4)}


def per_client_metrics(df, target_col="y", pred_col="iTransformer"):
    rows = []
    for uid, grp in df.groupby("unique_id"):
        rows.append({"client_id": uid, **compute_metrics(grp, target_col, pred_col)})
    rows.append({"client_id": "OVERALL", **compute_metrics(df, target_col, pred_col)})
    return pd.DataFrame(rows)

In [9]:
val_h = val["ds"].nunique()
print(f"Validation horizon (val_h): {val_h} hours")

# Fit on train + val combined, using val as internal validation for early stopping
train_val_nf = pd.concat([train, val], ignore_index=True).sort_values(
    ["unique_id", "ds"]).reset_index(drop=True)

model = iTransformer(
    h=val_h,
    input_size=INPUT_SIZE,
    n_series=N_SERIES,
    hidden_size=512,
    n_heads=8,
    e_layers=2,
    d_layers=1,
    d_ff=2048,
    factor=1,
    dropout=0.1,
    use_norm=True,
    loss=MSE(),
    valid_loss=MAE(),
    max_steps=MAX_STEPS,
    learning_rate=LEARNING_RATE,
    early_stop_patience_steps=EARLY_STOP,
    val_check_steps=VAL_CHECK,
    batch_size=BATCH_SIZE,
    scaler_type=SCALER_TYPE,
    random_seed=RANDOM_SEED,
)

nf = NeuralForecast(models=[model], freq="h")
nf.fit(df=train_val_nf, val_size=val_h)

nf.save(str(MODEL_DIR / "itransformer_val"))
print("Training complete. Model saved -> models/itransformer_val/")

Validation horizon (val_h): 2856 hours


INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name          ┃ Type                   ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss          │ MSE                    │      0 │ train │     0 │
│ 1 │ valid_loss    │ MAE                    │      0 │ train │     0 │
│ 2 │ padder_train  │ ConstantPad1d          │      0 │ train │     0 │
│ 3 │ scaler        │ TemporalNorm           │      0 │ train │     0 │
│ 4 │ enc_embedding │ DataEmbedding_inverted │  344 K │ train │     0 │
│ 5 │ encoder       │ TransEncoder           │  6.3 M │ train │     0 │
│ 6 │ projector     │ Linear                 │  1.5 M │ train │     0 │
└───┴───────────────┴────────────────────────┴────────┴───────┴───────┘

Trainable params: 8.1 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 8.1 M                                                                                                
Total estimated model params size (MB): 32                                                                         
Modules in train mode: 37                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

Training complete. Model saved -> models/itransformer_val/


In [11]:
# VALIDATION FORECAST
# NeuralForecast.predict() produces h-step-ahead forecasts from the end of the
# training portion.  futr_df supplies future exogenous features for the
# forecast horizon.

val_preds = nf.predict(df=train)
val_preds = val_preds.reset_index()

# Ensure matching datetime types
val_preds["ds"] = pd.to_datetime(val_preds["ds"])
val["ds"] = pd.to_datetime(val["ds"])

val_eval = val[["unique_id", "ds", "y"]].merge(val_preds, on=["unique_id", "ds"])
print(val_eval.head())

INFO:pytorch_lightning.utilities.rank_zero:Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

  unique_id                  ds          y  index  iTransformer
0    MT_124 2014-01-01 00:00:00  95.693780      0    117.284943
1    MT_124 2014-01-01 01:00:00  83.732056      1     97.090027
2    MT_124 2014-01-01 02:00:00  82.535890      2    100.107681
3    MT_124 2014-01-01 03:00:00  82.535890      3    127.166214
4    MT_124 2014-01-01 04:00:00  63.397133      4    166.892197


In [13]:
# TRAINING METRICS (In-Sample via Cross-Validation) + VALIDATION METRICS (Out-of-Sample)

# 1. Training Metrics — use cross_validation to get "in-sample" predictions
#    predict_insample() is NOT supported for multivariate models like iTransformer.
#    Instead, we run nf.predict() on a truncated training set as a proxy.

# Use only the last portion of training data as "in-sample" check:
# Feed all but the last val_h rows of training data, then predict the last val_h rows.
train_h = val_h  # use same horizon for comparability
cutoff = train.groupby("unique_id")["ds"].apply(
    lambda s: s.sort_values().iloc[-train_h]
).min()

train_hist = train[train["ds"] < cutoff].reset_index(drop=True)
train_future = train[train["ds"] >= cutoff].reset_index(drop=True)

train_preds = nf.predict(df=train_hist)
train_preds = train_preds.reset_index()
train_preds["ds"] = pd.to_datetime(train_preds["ds"])
train_future["ds"] = pd.to_datetime(train_future["ds"])

train_eval = train_future[["unique_id", "ds", "y"]].merge(
    train_preds, on=["unique_id", "ds"], how="inner")

train_metrics = per_client_metrics(train_eval.dropna())
print("-- Training Metrics (In-Sample proxy via predict on held-out tail) --")
print(train_metrics.head(5).to_string(index=False))
print("...")
print(train_metrics.tail(1).to_string(index=False))
print()

# 2. Validation Metrics (Out-of-Sample)
val_metrics = per_client_metrics(val_eval)
print("-- Validation Metrics (Out-of-Sample) --")
print(val_metrics.head(5).to_string(index=False))
print("...")
print(val_metrics.tail(1).to_string(index=False))

INFO:pytorch_lightning.utilities.rank_zero:Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

-- Training Metrics (In-Sample proxy via predict on held-out tail) --
client_id        MSE      MAE   WAPE
   MT_124 22965.3731 129.1322 0.4738
   MT_132   227.1077  11.4966 0.7873
   MT_156  1760.9968  32.2288 0.3838
   MT_158  3124.0525  46.6556 0.5841
   MT_159  2041.0787  38.0002 0.6489
...
client_id        MSE      MAE   WAPE
  OVERALL 530741.087 183.1086 0.2769

-- Validation Metrics (Out-of-Sample) --
client_id        MSE      MAE   WAPE
   MT_124 20340.4421 124.3682 0.4593
   MT_132   451.6303  16.9912 0.9936
   MT_156   756.8294  22.9094 0.2806
   MT_158  3925.2902  51.8945 0.7033
   MT_159  3103.3447  44.3223 0.8706
...
client_id         MSE     MAE   WAPE
  OVERALL 109725.3185 92.9169 0.1515


In [14]:
# TEST EVALUATION
# Retrain on full train+val (no held-out val), then predict on test set.

test_h = test["ds"].nunique()
print(f"Test horizon (test_h): {test_h} hours")

model_final = iTransformer(
    h=test_h,
    input_size=INPUT_SIZE,
    n_series=N_SERIES,
    hidden_size=512,
    n_heads=8,
    e_layers=2,
    d_layers=1,
    d_ff=2048,
    factor=1,
    dropout=0.1,
    use_norm=True,
    loss=MSE(),
    valid_loss=MAE(),
    max_steps=MAX_STEPS,
    learning_rate=LEARNING_RATE,
    early_stop_patience_steps=-1,  # no early stopping for final model
    batch_size=BATCH_SIZE,
    scaler_type=SCALER_TYPE,
    random_seed=RANDOM_SEED,
)

nf_final = NeuralForecast(models=[model_final], freq="h")
nf_final.fit(df=train_val_nf)  # no val_size -> uses all data for training

nf_final.save(str(MODEL_DIR / "itransformer_final"))
print("Final model saved -> models/itransformer_final/")

# Predict on test
test_preds = nf.predict(df=train_val_nf)
test_preds = test_preds.reset_index()

test_eval = test[["unique_id", "ds", "y"]].merge(test_preds, on=["unique_id", "ds"])

test_metrics = per_client_metrics(test_eval)
print("-- Test Metrics --")
print(test_metrics.to_string(index=False))

-- Test Metrics --
client_id          MSE       MAE   WAPE
   MT_124 4.615875e+03   54.3596 0.1746
   MT_132 2.241195e+02   11.3704 0.7806
   MT_156 5.794423e+02   18.0975 0.1989
   MT_158 1.636768e+03   34.1046 0.4806
   MT_159 9.544299e+02   24.9141 0.7217
   MT_161 8.084891e+04  230.5643 0.1522
   MT_162 1.826743e+04  109.3738 0.4357
   MT_163 2.272787e+05  386.7993 0.1479
   MT_166 2.209516e+04  116.4463 0.0881
   MT_168 4.766096e+02   18.3429 0.1391
   MT_169 4.443720e+01    5.2143 0.1223
   MT_171 1.458038e+03   30.6644 0.1267
   MT_172 5.378105e+02   17.8908 0.1461
   MT_174 1.178422e+03   26.5152 0.1561
   MT_175 1.263594e+03   28.2000 0.1719
   MT_176 3.601052e+02   15.3024 0.1701
   MT_180 1.176410e+03   26.5812 0.1685
   MT_182 8.779870e+01    7.6252 0.1479
   MT_183 3.768300e+02   16.2958 0.2084
   MT_187 4.439400e+02   16.9638 0.1769
   MT_188 4.342830e+01    5.3474 0.1221
   MT_189 3.008062e+03   42.3599 0.1050
   MT_190 2.814922e+03   37.8819 0.1059
   MT_191 7.374519e+0

In [15]:
import shutil
from google.colab import files

# Zip the directories
shutil.make_archive("/content/itransformer_final", 'zip', "/content/models/itransformer_final")
shutil.make_archive("/content/itransformer_val", 'zip', "/content/models/itransformer_val")

print("Zipping complete.")

Zipping complete.


In [16]:
# Download the zip files
files.download("/content/itransformer_final.zip")
files.download("/content/itransformer_val.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>